In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import random
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torchvision import transforms
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import torchvision.models as models

In [ ]:
def load_images_from_folder(folder_path, image_size=(224, 224)):
    images = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith((".jpg", ".jpeg")):
                try:
                    img_path = os.path.join(root, file)
                    img = Image.open(img_path).convert("RGB")
                    img = img.resize(image_size)
                    images.append(np.array(img))
                except Exception as e:
                    print(f"Failed on {img_path}: {e}")
    return np.array(images)

def plot_rgb_histogram_subplot(ax, images, class_name):
    sample = images[random.randint(0, len(images) - 1)]
    colors = ('r', 'g', 'b')
    for i, col in enumerate(colors):
        hist = np.histogram(sample[:, :, i], bins=256, range=(0, 256))[0]
        ax.plot(hist, color=col)
    ax.set_title(f"RGB Histogram – {class_name.capitalize()}")
    ax.set_xlabel("Pixel Value")
    ax.set_ylabel("Frequency")

def augment_rotations(X, y):
    X_aug = []
    y_aug = []
    for k in [1, 2, 3]: 
        X_rot = torch.rot90(X, k=k, dims=[2, 3])  
        X_aug.append(X_rot)
        y_aug.append(y.clone()) 
    return torch.cat(X_aug), torch.cat(y_aug)

In [ ]:
strawberry_halved = "dataset/Strawberry_512/Hulled"
strawberry_sliced = "dataset/Strawberry_512/Sliced"
strawberry_whole = "dataset/Strawberry_512/Whole"

In [ ]:


strawberry_hulled_images = load_images_from_folder(strawberry_halved)
strawberry_sliced_images = load_images_from_folder(strawberry_sliced)
strawberry_whole_images = load_images_from_folder(strawberry_whole)

print("Strawberry halved images:", strawberry_hulled_images.shape)
print("Strawberry sliced images:", strawberry_sliced_images.shape)
print("Strawberry whole images:", strawberry_whole_images.shape)


In [ ]:
import matplotlib.pyplot as plt
import random
datasets = {
    "Hulled": strawberry_hulled_images,
    "sliced": strawberry_sliced_images,
    "whole": strawberry_whole_images
}


def show_random_samples(images, class_name, count=5):
    indices = random.sample(range(images.shape[0]), count)
    selected = images[indices]

    plt.figure(figsize=(10, 2))
    for i, img in enumerate(selected):
        plt.subplot(1, count, i+1)
        plt.imshow(img.astype(np.uint8))
        plt.axis('off')
    plt.suptitle(f"{class_name.capitalize()} – Random {count} Samples", fontsize=16)
    plt.show()

for class_name, image_array in datasets.items():
    show_random_samples(image_array, class_name)


In [ ]:
fig, axes = plt.subplots(1, len(datasets), figsize=(20, 5))

for ax, (class_name, images) in zip(axes, datasets.items()):
    plot_rgb_histogram_subplot(ax, images, class_name)
    ax.label_outer() 

plt.tight_layout()
plt.show()

In [ ]:
class_names = list(datasets.keys())
num_classes = len(class_names)

fig, axes = plt.subplots(1, num_classes, figsize=(4 * num_classes, 4)) 

for i, (class_name, images) in enumerate(datasets.items()):
    avg_img = np.mean(images.astype(np.float32), axis=0)
    axes[i].imshow(avg_img.astype(np.uint8))
    axes[i].set_title(f"Average Image – {class_name.capitalize()}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
datasets = {
    "hulled": strawberry_hulled_images,
    "sliced": strawberry_sliced_images,
    "whole": strawberry_whole_images
}

X = np.concatenate([strawberry_hulled_images, strawberry_sliced_images, strawberry_whole_images], axis=0)
y = (
    ['hulled'] * len(strawberry_hulled_images) +
    ['sliced'] * len(strawberry_sliced_images) +
    ['whole'] * len(strawberry_whole_images)
)

X = X.astype(np.float32) / 255.0
X = np.transpose(X, (0, 3, 1, 2)) 
X_tensor = torch.tensor(X)

le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_tensor = torch.tensor(y_encoded)

X_train, X_temp, y_train, y_temp = train_test_split(X_tensor, y_tensor, test_size=0.5, stratify=y_tensor, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)


In [ ]:
batch_size = 32

X_augmented, y_augmented = augment_rotations(X_train, y_train)

X_train_combined = torch.cat([X_train, X_augmented])
y_train_combined = torch.cat([y_train, y_augmented])

train_dataset = TensorDataset(X_train_combined, y_train_combined)
val_dataset   = TensorDataset(X_val, y_val)
test_dataset  = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size)
test_loader  = DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
print(f"Train Dataset: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"Val Dataset:   {len(val_dataset)} samples, {len(val_loader)} batches")
print(f"Test Dataset:  {len(test_dataset)} samples, {len(test_loader)} batches")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

import torch.nn as nn
import torchvision.models as models

def get_efficientnet_model(num_classes):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model



In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple GPU)")
else:
    device = torch.device("cpu")
    print("MPS not available. Using CPU")

model = get_efficientnet_model(num_classes=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
best_val_acc = 0.0
train_losses = []
val_losses = []
train_accs = []
val_accs = []
epochs_no_improve = 0
early_stop = False
patience = 6
model_name = "models/best_model_strawberry_v1.pth"

for epoch in range(30):
    if early_stop:
        print(f"Early stopping at epoch {epoch}")
        break
    model.train()
    total_train_loss = 0
    train_correct = 0
    train_total = 0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        preds = model(batch_x)
        loss = criterion(preds, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

        pred_labels = preds.argmax(dim=1)
        train_correct += (pred_labels == batch_y).sum().item()
        train_total += batch_y.size(0)

    train_accuracy = train_correct / train_total
    avg_train_loss = total_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    train_accs.append(train_accuracy)

    
    model.eval()
    val_correct = val_total = 0

    with torch.no_grad():
        for val_x, val_y in val_loader:
            val_x, val_y = val_x.to(device), val_y.to(device)
            val_preds = model(val_x).argmax(dim=1)
            val_correct += (val_preds == val_y).sum().item()
            val_total += val_y.size(0)

    val_accuracy = val_correct / val_total
    validation_loss = criterion(model(val_x), val_y).item()

    val_losses.append(validation_loss)
    val_accs.append(val_accuracy)

    print(f"Epoch {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | "
          f"Train Acc: {train_accuracy:.4f} | Val Acc: {val_accuracy:.4f}")
    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        torch.save(model.state_dict(), model_name)
        print(f"New best model saved at epoch {epoch+1} with val acc {val_accuracy:.4f}")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epoch(s)")

    if epochs_no_improve >= patience:
        print(f"Validation accuracy did not improve for {patience} consecutive epochs. Stopping early.")
        early_stop = True



In [ ]:
epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses, label='Train Loss', marker='o')
plt.plot(epochs, val_losses, label='Validation Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss per Epoch')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs, train_accs, label='Train Accuracy', marker='o')
plt.plot(epochs, val_accs, label='Validation Accuracy', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy per Epoch')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:

model = get_efficientnet_model(num_classes=3).to(device)
model.load_state_dict(torch.load(model_name))
model.eval()  

all_preds = []
all_targets = []
all_images = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        preds = model(batch_x).argmax(dim=1).cpu()
        all_preds.extend(preds.numpy())
        all_targets.extend(batch_y.numpy())
        all_images.extend(batch_x.cpu())

test_correct = sum(np.array(all_preds) == np.array(all_targets))
test_total = len(all_targets)
test_accuracy = test_correct / test_total

print(f"\nTest Accuracy: {test_accuracy:.4f}")

target_names = le.classes_
print("\nClassification Report:\n")
print(classification_report(all_targets, all_preds, target_names=target_names))

cm = confusion_matrix(all_targets, all_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=target_names, yticklabels=target_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
all_preds = np.array(all_preds)
all_targets = np.array(all_targets)
all_images = torch.stack(all_images) 

for class_idx, class_name in enumerate(target_names):
    print(f"\nShowing False Negatives and False Positives for class: {class_name}")
    fn_indices = np.where((all_targets == class_idx) & (all_preds != class_idx))[0]
    fp_indices = np.where((all_preds == class_idx) & (all_targets != class_idx))[0]

    def show_images(indices, title, max_images=5):
        num = min(len(indices), max_images)
        if num == 0:
            print(f"No {title} samples.")
            return

        plt.figure(figsize=(12, 2))
        for i, idx in enumerate(indices[:num]):
            img = all_images[idx]
            img = img.permute(1, 2, 0).numpy()
            plt.subplot(1, num, i + 1)
            plt.imshow((img - img.min()) / (img.max() - img.min()))
            plt.axis('off')
            plt.title(f"Pred: {target_names[all_preds[idx]]}\nTrue: {target_names[all_targets[idx]]}")
        plt.suptitle(f"{title} for {class_name}")
        plt.tight_layout()
        plt.show()

    show_images(fn_indices, "False Negatives")
    show_images(fp_indices, "False Positives")


In [ ]:
def visualize_channels(model, image_tensor, max_channels=6):
    model.eval()
    activations = {}

    def get_activation(name):
        def hook(model, input, output):
            activations[name] = output.detach().cpu()
        return hook

    hooks = []
    for i in range(len(model.features)):
        layer = model.features[i]
        hooks.append(layer.register_forward_hook(get_activation(f"features_{i}")))

    with torch.no_grad():
        _ = model(image_tensor.unsqueeze(0)) 

    for h in hooks:
        h.remove()

    for layer_name, fmap in activations.items():
        fmap = fmap.squeeze(0) 

        channel_scores = fmap.mean(dim=(1, 2))

        topk = torch.topk(channel_scores, k=min(max_channels, fmap.shape[0]))
        top_indices = topk.indices

        plt.figure(figsize=(max_channels * 2, 2.5))
        for idx, ch in enumerate(top_indices):
            plt.subplot(1, max_channels, idx + 1)
            plt.imshow(fmap[ch], cmap='viridis')
            plt.title(f"{layer_name}\nch{ch.item()} ({channel_scores[ch]:.2f})")
            plt.axis('off')
        plt.tight_layout()
        plt.show()


In [ ]:
model = get_efficientnet_model(num_classes=3)
model.load_state_dict(torch.load(model_name))
model.eval()

In [ ]:

img = Image.open("dataset/Strawberry_512/Whole/image_0017.jpg").convert("RGB")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])
img_tensor = transform(img)  

visualize_channels(model, img_tensor, max_channels=16)


In [ ]:

img = Image.open("dataset/Strawberry_512/Hulled/image_0001.jpg").convert("RGB")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])
img_tensor = transform(img)  

visualize_channels(model, img_tensor, max_channels=16)


In [ ]:

img = Image.open("dataset/Strawberry_512/Sliced/image_0001.jpg").convert("RGB")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])
img_tensor = transform(img)  

visualize_channels(model, img_tensor, max_channels=16)